# 🚀 THE QUANT — Google Colab Remote Training Worker

Run all cells to turn this Colab session into the GPU training backend for THE-QUANT.
The notebook:
1. Detects your accelerator (T4 / A100 / L4 / TPU / CPU)
2. Clones the latest **THE-QUANT** repo (real-data training pipeline incl. Dukascopy tick downloader)
3. Starts the token-secured worker server + public Cloudflare tunnel

> ⚠️ The tunnel URL is public — anyone with it could read served artifacts. Keep the
> runtime alive only while your local agent is connected, then stop the session.

In [ ]:
!pip install -q torch onnx onnxruntime scikit-learn xgboost pandas numpy requests cloudflared

import torch, sys, os
print('=' * 65)
print('THE QUANT RUNTIME ENVIRONMENT DETECTED:')
print(f'Python Version : {sys.version.split()[0]}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'⚡ GPU ACCELERATOR: {gpu_name} ({vram:.2f} GB VRAM)')
    print(f'CUDA Version    : {torch.version.cuda}')
else:
    print('💻 ACCELERATOR : CPU (Runtime ▸ Change runtime type ▸ T4 GPU recommended)')
print('=' * 65)

In [ ]:
import os, subprocess, pathlib

WORK = '/content/quant_colab'
REPO = WORK + '/repo'
os.makedirs(WORK, exist_ok=True)

if os.path.exists(REPO + '/config/system.toml'):
    # refresh to latest main
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', '--quiet'], capture_output=True)
    print('[+] repo already cloned; pulled latest main')
else:
    r = subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/elmaxadore/THE-QUANT.git', REPO],
                       capture_output=True, text=True)
    print(r.stdout[-2000:] or '[+] cloned THE-QUANT repo')

# the worker inherits this env -> the real-data training pipeline finds the repo
os.environ['COLAB_REPO_DIR'] = REPO

SECRET_TOKEN = os.environ.get('COLAB_TOKEN', 'quant-colab-secret-key')
PORT = 8095
print(f'[+] Worker dir  : {WORK}')
print(f'[+] Repo dir    : {REPO}')
print(f'[+] Secret Token: {SECRET_TOKEN}')

In [ ]:
import subprocess, time, re

# cloudflared binary (quick tunnel)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', '-nc',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'],
                   check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], capture_output=True)
print('[+] cloudflared ready')

worker_proc = subprocess.Popen([sys.executable, REPO + '/colab/colab_worker.py',
                                '--port', str(PORT), '--token', SECRET_TOKEN])
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
print('Waiting for Cloudflare Public Tunnel URL...')
for line in tunnel_proc.stdout:
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break

print()
print('=' * 70)
print('🚀 THE QUANT COLAB WORKER IS LIVE!')
print('=' * 70)
print(f'📍 Public Tunnel URL : {public_url}')
print(f'🔑 Secret Token      : {SECRET_TOKEN}')
print('=' * 70)
print()
print('👉 Paste these two values into your local agent/session.')
print('👉 Local harness:  python colab/colab_cli.py connect --url <URL> --token <TOKEN>')
print('👉 Full pipeline:  python colab/colab_cli.py train --type custom \')
print('                      --script python/research/train_pipeline.py \')
print('                      --params-json \'{"symbols": "eurusd,gbpusd,xauusd,audusd,nzdusd,xauusd,xagusd","start":"2023-01-01"}\'\')
print('=' * 70)